# Réplica controlada: *More Tokens, Fewer Trees* con Qwen3.5-4B en EXAMS-V (de principio a fin)

Este notebook ejecuta toda la réplica de [arXiv:2607.09438](https://arxiv.org/abs/2607.09438) con el código
oficial [lesterpjy/tts-small-vlm](https://github.com/lesterpjy/tts-small-vlm), en orden y de arriba hacia abajo:

| Paso | Qué hace | Tiempo aprox. (L40S) |
|---|---|---|
| 0 | Parámetros, verificación de GPU y disco, temporizador de seguridad | 1 min |
| 1 | Entorno: repo en el commit fijado, venv Python 3.12, modelo y datos en revisiones fijas | 5–15 min |
| 2 | Verificación de los datos y una pregunta de ejemplo | 1 min |
| 3 | Pruebas unitarias del repositorio | 1 min |
| 4 | Subconjunto estratificado de 200 preguntas (semilla 42) | 1 min |
| 5 | Prueba de humo: 10 preguntas × 3 configuraciones | 15 min |
| 6 | Experimentos: zero-shot, self-consistency N=8 con 1,024 y con 2,048 tokens | 1.5 h |
| 7 | Reparación guiada oficial sobre los resultados de 2,048 tokens | 3 min |
| 8 | Evaluación: todas las métricas recalculadas y verificadas desde los registros crudos | 1 min |
| 9 | Tablas comparativas, intervalos de confianza, comparación por idioma con el paper, ejemplos | 1 min |
| 10 | Pruebas otra vez, validación de entrega y escaneo del paquete | 1 min |
| 11 | Copia a S3 y apagado opcional de la app | 1 min |

**Dónde ejecutarlo.** Un espacio JupyterLab de SageMaker Studio en `ml.g6e.2xlarge` (1× L40S de 46 GB) con
200 GB de disco y el kernel *Python 3 (ipykernel)*. También sirve cualquier Linux con una GPU NVIDIA de 40 GB
o más. Las fases con GPU corren en un venv propio que crea el paso 1; este kernel solo las coordina.

**Costo.** Unas 2 h de instancia a $2.80/h, es decir unos $6. Un temporizador apaga la app al llegar a
`MAX_RUNTIME_H` horas.

**Cómo usarlo.** Copia la carpeta `replication/` al espacio (por ejemplo
`aws s3 sync s3://<bucket>/replication ~/work/replication` desde una terminal de Studio), abre este notebook,
revisa la celda de parámetros y usa *Run → Run All Cells*.

**Reanudación.** Con `RESUME = True`, se salta cada fase cuyo `metrics.json` ya existe y pasa todas las
verificaciones; si el kernel se reinicia, *Run All* continúa donde se quedó. Con `RESUME = False` se regenera todo.

**Reglas que respeta.** El código del repositorio oficial no se modifica. El modelo queda congelado: no hay
entrenamiento. Se usan los prompts oficiales, `generate_n` de vLLM (una llamada por pregunta que devuelve las
N cadenas), `extract_answer`, `majority_vote` y `eval/evaluate.py`. Las predicciones no se editan a mano.
Toda la lógica vive en `scripts/`; este notebook llama a esos scripts, así que los comandos del `README.md`
y el notebook producen exactamente los mismos archivos.

## 0. Parámetros

In [ ]:
import os
from pathlib import Path

# --- Qué ejecutar -----------------------------------------------------------
RUN_SETUP = True          # crear o actualizar el venv y la caché de Hugging Face (una vez por espacio)
RUN_TESTS = True          # pruebas unitarias oficiales, antes y después
RUN_GPU_STAGES = True     # prueba de humo, experimentos y reparación guiada
RUN_REPAIR = True         # reparación guiada sobre 2k (se salta sola si ninguna pregunta quedó sin letra)
RESUME = True             # saltar fases con un metrics.json ya verificado
SHOW_EXAMPLE_IMAGE = True # mostrar la imagen de una pregunta de ejemplo (no se guarda en el paquete)

# --- Tamaños (200 / 10 para reproducir los resultados reportados) ---------
SUBSET_SIZE = 200
SMOKE_SIZE = 10

# --- Control de costo ---------------------------------------------------------
MAX_RUNTIME_H = 8.0         # temporizador: borra esta app de Studio al pasar estas horas
S3_BUCKET = ""              # p. ej. "mi-bucket-privado"; vacío = no copiar el paquete a S3
S3_PREFIX = "replication"
STOP_APP_WHEN_DONE = False  # borrar esta app al final (solo si la copia a S3 salió bien)

# --- Rutas ----------------------------------------------------------------------
REPL = Path.cwd().resolve()   # este paquete (la carpeta que contiene el notebook)
WORK = Path("/home/sagemaker-user/work") if Path("/home/sagemaker-user").exists() else REPL.parent / "work"

# Variables de entorno NB_* para ejecutar sin interfaz (por ejemplo con nbconvert)
for _name in ["RUN_SETUP", "RUN_TESTS", "RUN_GPU_STAGES", "RUN_REPAIR", "RESUME", "STOP_APP_WHEN_DONE",
              "SHOW_EXAMPLE_IMAGE"]:
    if f"NB_{_name}" in os.environ:
        globals()[_name] = os.environ[f"NB_{_name}"] == "1"
S3_BUCKET = os.environ.get("NB_S3_BUCKET", S3_BUCKET)
S3_PREFIX = os.environ.get("NB_S3_PREFIX", S3_PREFIX)
MAX_RUNTIME_H = float(os.environ.get("NB_MAX_RUNTIME_H", MAX_RUNTIME_H))
WORK = Path(os.environ.get("WORK", WORK))

### Funciones auxiliares

`sh()` ejecuta un comando y muestra su salida en vivo; en las fases largas imprime cada minuto cuántas preguntas
van terminadas. `stage()` registra el resultado de cada fase; un error nunca se oculta: se anota y detiene el
notebook.

In [ ]:
import json, re, shutil, subprocess, sys, threading, time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

assert (REPL / "scripts" / "env.sh").exists(), f"abre este notebook desde la carpeta replication/ (cwd={REPL})"
os.environ.update({
    "REPL": str(REPL), "WORK": str(WORK),
    "REPO": os.environ.get("REPO", str(WORK / "tts-small-vlm")),
    "VENV": os.environ.get("VENV", str(WORK / "venv")),
    "HF_HOME": os.environ.get("HF_HOME", str(WORK / "hf-cache")),
    "SUBSET_SIZE": str(SUBSET_SIZE), "SMOKE_SIZE": str(SMOKE_SIZE),
    "SYSPY": sys.executable,      # intérprete que instala uv (fuera del venv)
})
WORK.mkdir(parents=True, exist_ok=True)
NB_START = time.time()
STATUS = []   # una fila por fase: nombre, estado, segundos, nota


def utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def check_budget():
    # Detenerse antes de empezar una fase si ya se agotó el presupuesto de tiempo
    used_h = (time.time() - NB_START) / 3600
    if used_h > MAX_RUNTIME_H:
        raise RuntimeError(f"presupuesto de tiempo agotado ({used_h:.2f} h > {MAX_RUNTIME_H} h)")


def sh(cmd, *, check=True, progress_log=None, every=60):
    # Ejecutar `cmd` con bash desde la raíz del paquete y mostrar la salida en vivo.
    # Si se da `progress_log`, cada `every` segundos se imprime cuántas preguntas van terminadas.
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(["bash", "-c", cmd], cwd=REPL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    reader = threading.Thread(target=lambda: [print(line, end="", flush=True) for line in proc.stdout])
    reader.start()
    t0 = last = time.time()
    while proc.poll() is None:
        time.sleep(2)
        if progress_log and time.time() - last >= every:
            last = time.time()
            p = Path(progress_log)
            done = len(re.findall(r"\] gold=", p.read_text(errors="replace"))) if p.exists() else 0
            print(f"  … {int(time.time() - t0)} s transcurridos, {done} preguntas terminadas", flush=True)
    reader.join()
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode


def stage(name, fn, *, enabled=True, note_disabled="desactivada por parámetro"):
    # Ejecutar una fase y registrar el resultado. Las excepciones se anotan y se relanzan (nunca se ocultan).
    if not enabled:
        STATUS.append({"stage": name, "status": "skipped", "seconds": 0, "note": note_disabled})
        print(f"[{name}] saltada: {note_disabled}")
        return None
    check_budget()
    t0 = time.time()
    print(f"[{name}] inicio {utc()}")
    try:
        note = fn() or ""
    except Exception as e:
        STATUS.append({"stage": name, "status": "FAILED", "seconds": round(time.time() - t0, 1), "note": repr(e)})
        raise
    STATUS.append({"stage": name, "status": "ok", "seconds": round(time.time() - t0, 1), "note": note})
    print(f"[{name}] ok en {time.time() - t0:.0f} s {note}")


def metrics(name):
    p = REPL / "results" / name / "metrics.json"
    return json.loads(p.read_text()) if p.exists() else None


def verified(name, expected):
    # Una fase cuenta como hecha solo si su metrics.json existe, pasó todas las verificaciones y tiene N preguntas
    m = metrics(name)
    return bool(m and m["verification"]["all_checks_passed"] and m["total"] == expected)


print("REPL =", REPL); print("WORK =", WORK); print("inicio", utc())

### Verificación de GPU y disco

Antes de gastar tiempo de instancia se comprueba que haya una GPU de al menos 40 GB y al menos 100 GB libres
(modelo ~9 GB, dataset ~8 GB, venv ~15 GB).

In [ ]:
# Requisitos mínimos de hardware (solo si se van a ejecutar las fases con GPU)
if RUN_GPU_STAGES:
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
    assert q.returncode == 0, "nvidia-smi no responde: esta máquina no tiene GPU NVIDIA visible"
    name, mem_mib, driver = [x.strip() for x in q.stdout.splitlines()[0].split(",")]
    print(f"GPU: {name}, {int(mem_mib)} MiB, driver {driver}")
    assert int(mem_mib) >= 40000, "se necesita una GPU con al menos 40 GB"
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"Disco libre en WORK: {free_gb:.0f} GB")
assert not RUN_SETUP or free_gb > 100, "se recomiendan al menos 100 GB libres"

### Temporizador de seguridad (solo en SageMaker Studio)

Si el notebook corre en una app de Studio, un temporizador en segundo plano borra la app al cumplirse
`MAX_RUNTIME_H` horas, lo que detiene el cobro de la instancia. Antes copia el paquete a S3 si `S3_BUCKET` está
definido. Usa el rol de ejecución del espacio (`sagemaker:DeleteApp` sobre su propia app); no hay credenciales
en el notebook.

In [ ]:
META = Path("/opt/ml/metadata/resource-metadata.json")


def sync_to_s3():
    # Copiar el paquete a S3 con el rol del espacio (sin credenciales en el código)
    if not S3_BUCKET:
        print("S3_BUCKET vacío; no se copia a S3"); return False
    os.environ["REPL_BUCKET"] = S3_BUCKET
    sh(f'"{sys.executable}" scripts/aws/s3sync.py up "{REPL}" "{S3_PREFIX}"')
    return True


def stop_this_app(reason):
    # Borrar esta app de JupyterLab (detiene el cobro de la instancia; el disco del espacio se conserva)
    import boto3
    m = json.loads(META.read_text())
    print(f"borrando la app de Studio {m['SpaceName']}/{m.get('ResourceName', 'default')} ({reason})", flush=True)
    boto3.client("sagemaker").delete_app(DomainId=m["DomainId"], SpaceName=m["SpaceName"],
                                         AppType=m.get("AppType", "JupyterLab"),
                                         AppName=m.get("ResourceName", "default"))


def _watchdog():
    try:
        sync_to_s3()
    finally:
        stop_this_app("se alcanzó MAX_RUNTIME_H")


if META.exists():
    WATCHDOG = threading.Timer(MAX_RUNTIME_H * 3600, _watchdog)
    WATCHDOG.daemon = True
    WATCHDOG.start()
    print(f"temporizador activo: la app se borrará a las "
          f"{datetime.fromtimestamp(NB_START + MAX_RUNTIME_H * 3600, timezone.utc):%Y-%m-%d %H:%M} UTC")
else:
    WATCHDOG = None
    print("no se está en SageMaker Studio; temporizador no activado")

## 1. Creación del entorno

`scripts/setup_env.sh` clona el repositorio y fija el commit `2247c130…` (si algún día no existiera, usa `main`
y deja el SHA elegido en `environment/repo-commit.txt`). Crea un venv con Python 3.12, instala
`requirements.txt` con uv y escribe `environment/requirements-lock.txt`. Descarga
`Qwen/Qwen3.5-4B@851bf6e8…` y `MBZUAI/EXAMS-V@7594b37a…`, y guarda `environment/system-info.txt`. Después,
los experimentos corren sin conexión contra exactamente esas revisiones.

Ambos repositorios de Hugging Face son públicos, así que no hace falta token.

In [ ]:
# Instalación completa (idempotente: si el venv y la caché ya existen, termina rápido)
stage("setup", lambda: sh("bash scripts/setup_env.sh"), enabled=RUN_SETUP)
if (REPL / "environment" / "system-info.txt").exists():
    print((REPL / "environment" / "system-info.txt").read_text())

## 2. Verificación de los datos y pregunta de ejemplo

Se comprueba la revisión fijada de EXAMS-V: 4,651 preguntas en `validation`, columnas e idiomas. En EXAMS-V el
enunciado y las opciones vienen dentro de la imagen; la celda muestra una para que se entienda la tarea. La imagen
solo se muestra en el notebook: no se guarda ningún archivo del dataset en el paquete.

In [ ]:
# Conteo de filas, columnas e idiomas del split de validación, con la revisión en caché (sin red)
CHECK = r'''
import json, os
from collections import Counter
from datasets import load_dataset
ds = load_dataset(os.environ["DATASET_ID"])["validation"]
print(json.dumps({"rows": ds.num_rows, "columns": ds.column_names,
                  "languages": dict(Counter(ds["language"]).most_common())}, ensure_ascii=False))
'''


def check_data():
    out = subprocess.run(["bash", "-c", "source scripts/env.sh && HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 python -"],
                         input=CHECK, cwd=REPL, capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout[-2000:], out.stderr[-3000:]); raise RuntimeError("no se pudo leer EXAMS-V")
    info = json.loads(out.stdout.strip().splitlines()[-1])
    print(f"filas: {info['rows']}\ncolumnas: {info['columns']}\nidiomas: {info['languages']}")
    assert info["rows"] == 4651, f"se esperaban 4,651 preguntas en validation, hay {info['rows']}"
    return f"{info['rows']} preguntas, {len(info['languages'])} idiomas"


stage("data-check", check_data, enabled=RUN_GPU_STAGES, note_disabled="sin fases de GPU (no hace falta el dataset)")

In [ ]:
# Mostrar una pregunta de ejemplo (la primera del subconjunto, si ya existe; si no, la fila 0)
SHOW = r'''
import io, json, os, sys
from datasets import load_dataset
ds = load_dataset(os.environ["DATASET_ID"])["validation"]
i = int(sys.argv[1]); r = ds[i]
img = r["image"].convert("RGB"); img.thumbnail((900, 900))
img.save(sys.argv[2], format="PNG")
print(json.dumps({"index": i, "sample_id": r["sample_id"], "language": r["language"],
                  "subject": r["subject"], "answer_key": r["answer_key"]}, ensure_ascii=False))
'''
if SHOW_EXAMPLE_IMAGE and RUN_GPU_STAGES:
    import tempfile
    from IPython.display import Image, display
    _m = REPL / "subset" / "subset-manifest.json"
    _idx = json.loads(_m.read_text())["indices"][0] if _m.exists() else 0
    with tempfile.TemporaryDirectory() as _tmp:   # la imagen vive solo en /tmp y se borra al salir
        _png = Path(_tmp) / "ejemplo.png"
        out = subprocess.run(["bash", "-c", f"source scripts/env.sh && HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 python - {_idx} {_png}"],
                             input=SHOW, cwd=REPL, capture_output=True, text=True)
        if out.returncode == 0:
            print(out.stdout.strip().splitlines()[-1])
            display(Image(filename=str(_png), width=600))
        else:
            print("no se pudo mostrar la imagen de ejemplo:", out.stderr[-500:])

## 3. Pruebas unitarias del repositorio (antes de los experimentos)

En este commit falla una prueba conocida: `tests/test_utils.py::TestSaveResults::test_save_competition_format`.
Espera la clave `prediction`, que `save_results` no escribe a propósito (su docstring lo explica), y ninguna parte
de esta réplica usa esa función. Se reporta, no se corrige. Cualquier otra falla detiene el notebook.

In [ ]:
KNOWN_FAILURES = {"tests.test_utils.TestSaveResults::test_save_competition_format"}


def run_tests(label):
    # Ejecutar pytest y leer el XML de JUnit para distinguir la falla conocida de cualquier otra
    sh(f"bash scripts/run_tests.sh {label}", check=False)
    root = ET.parse(REPL / "logs" / f"tests-{label}.junit.xml").getroot()
    cases = list(root.iter("testcase"))
    failed = {f"{c.get('classname')}::{c.get('name')}" for c in cases
              if c.find("failure") is not None or c.find("error") is not None}
    unexpected = failed - KNOWN_FAILURES
    print(f"{len(cases)} pruebas, {len(failed)} fallidas, fallas inesperadas: {sorted(unexpected) or 'ninguna'}")
    if unexpected:
        raise RuntimeError(f"fallas inesperadas en las pruebas: {sorted(unexpected)}")
    return f"{len(cases) - len(failed)} pasan, {len(failed)} falla conocida"


stage("tests-before", lambda: run_tests("before"), enabled=RUN_TESTS)

## 4. Subconjunto estratificado (semilla 42)

`scripts/make_subset_manifest.py` llama a la función oficial `src.utils.stratified_indices` igual que
`scripts/experiment.py` (200 preguntas, semilla 42, `min_per_language=10`). Escribe
`subset/subset-manifest.json` (índices, asignación por estrato, ids y respuestas correctas) y
`subset/references.json` (para el evaluador oficial). Después se verifica que cada corrida use exactamente
este subconjunto.

In [ ]:
def make_subset():
    if RESUME and (REPL / "subset" / "subset-manifest.json").exists():
        return "se conserva el manifiesto existente (RESUME=True)"
    sh("source scripts/env.sh && HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 python scripts/make_subset_manifest.py")


stage("subset", make_subset)
MANIFEST = json.loads((REPL / "subset" / "subset-manifest.json").read_text())
assert MANIFEST["seed"] == 42 and MANIFEST["subset_size"] == SUBSET_SIZE, "el manifiesto no coincide con los parámetros"
print(f"{MANIFEST['subset_size']} preguntas, semilla {MANIFEST['seed']}, {len(MANIFEST['allocation'])} estratos")
print(json.dumps(MANIFEST["language_counts"], ensure_ascii=False))

## 5–6. Prueba de humo y experimentos

Cada configuración pasa sin cambios por el ejecutor oficial:
`python $REPO/scripts/experiment.py --config $REPO/<config> --subset-override 200`
(con `--end-idx 10` en la prueba de humo). `scripts/run_experiment.sh` además muestrea la memoria de la GPU con
nvidia-smi y guarda tiempo total y código de salida. Luego `scripts/compute_metrics.py` recalcula y verifica
todas las métricas desde los registros guardados; si alguna verificación falla, la fase falla.

Duración esperada en una L40S: zero-shot ~4 min, SC-1k ~37 min, SC-2k ~50 min, prueba de humo ~15 min.

In [ ]:
# Las tres configuraciones oficiales que pide la tarea
CONFIGS = {
    "zero-shot": "configs/baselines/q35_zero_shot.yaml",
    "self-consistency-1k": "configs/scaling/q35_sc_n8_1k.yaml",
    "self-consistency-2k": "configs/scaling/q35_sc_n8_2k.yaml",
}


def experiment(name, config, *, smoke=False):
    expected = SMOKE_SIZE if smoke else SUBSET_SIZE
    if RESUME and verified(name, expected):
        return "ya verificada (RESUME=True)"
    extra = f"--end-idx {SMOKE_SIZE}" if smoke else ""
    # Ejecutor oficial + muestreo de GPU
    sh(f"bash scripts/run_experiment.sh {name} {config} {extra}",
       progress_log=REPL / "results" / name / "experiment.log")
    # Métricas y verificaciones desde los registros crudos
    sh(f"source scripts/env.sh && python scripts/compute_metrics.py results/{name}"
       + (f" --expected {SMOKE_SIZE}" if smoke else ""))
    m = metrics(name)
    if not m["verification"]["all_checks_passed"]:
        raise RuntimeError(f"{name}: verificación fallida: {json.dumps(m['verification'], indent=1)}")
    lo, hi = m["accuracy_ci95_wilson"]
    return f"accuracy {m['accuracy']:.3f} ({m['correct']}/{m['total']}), IC95 [{lo:.3f}, {hi:.3f}]"

### 5. Prueba de humo (primeras 10 preguntas del subconjunto, las tres configuraciones)

In [ ]:
for _name, _cfg in CONFIGS.items():
    stage(f"smoke/{_name}", lambda n=_name, c=_cfg: experiment(f"smoke-test/{n}", c, smoke=True),
          enabled=RUN_GPU_STAGES)

In [ ]:
# Inspeccionar una pregunta de la prueba de humo: 8 cadenas, letras extraídas y votación
def first_record(name):
    f = next((REPL / "results" / name / "raw").glob("*/candidates.jsonl"))
    return json.loads(f.open().readline())


if (REPL / "results" / "smoke-test" / "self-consistency-2k" / "raw").exists():
    r = first_record("smoke-test/self-consistency-2k")
    print("idioma:", r["language"], "| materia:", r["subject"], "| respuesta correcta:", r["gold"])
    print("letras de las 8 cadenas:", [c.get("extracted_answer") for c in r["chains"]])
    print("votos:", r["verifier"]["cluster_sizes"], "| elegida:", r["verifier"]["selected_answer"],
          "| correcta:", r["correct"])
    print("\n--- final de la primera cadena ---\n", r["chains"][0]["reasoning"][-600:])

### 6a. Qwen3.5-4B zero-shot: `configs/baselines/q35_zero_shot.yaml`

In [ ]:
stage("zero-shot", lambda: experiment("zero-shot", CONFIGS["zero-shot"]), enabled=RUN_GPU_STAGES)

### 6b. Self-consistency N=8, T=0.7, 1,024 tokens: `configs/scaling/q35_sc_n8_1k.yaml`

In [ ]:
stage("self-consistency-1k", lambda: experiment("self-consistency-1k", CONFIGS["self-consistency-1k"]),
      enabled=RUN_GPU_STAGES)

### 6c. Self-consistency N=8, T=0.7, 2,048 tokens: `configs/scaling/q35_sc_n8_2k.yaml`

In [ ]:
stage("self-consistency-2k", lambda: experiment("self-consistency-2k", CONFIGS["self-consistency-2k"]),
      enabled=RUN_GPU_STAGES)

## 7. Reparación guiada oficial (resultados de 2,048 tokens)

`scripts/run_guided_repair.sh` copia los registros de SC-2k a `results/guided-repair/raw/` y ejecuta el script
oficial `scripts/repair_parse_failures.py` sobre la copia; los originales no se tocan. Primero hace una corrida
en seco. El script oficial repara solo las preguntas en las que ninguna cadena dio una letra: añade
`"\n\nAnswer: "` a cada cadena, fuerza una letra A–E con decodificación guiada y vota entre esas letras.

In [ ]:
def repair():
    sc2k = metrics("self-consistency-2k")
    if sc2k["answer_extraction"]["question_level_failures"] == 0:
        return "ninguna pregunta quedó sin letra; no hace falta reparar"
    if RESUME and verified("guided-repair", SUBSET_SIZE):
        return "ya verificada (RESUME=True)"
    sh("bash scripts/run_guided_repair.sh")
    sh("source scripts/env.sh && python scripts/compute_metrics.py results/guided-repair")
    m = metrics("guided-repair")
    if not m["verification"]["all_checks_passed"]:
        raise RuntimeError("guided-repair: verificación fallida")
    return f"{m['guided_repair']['questions_repaired']} preguntas reparadas; accuracy {m['accuracy']:.3f}"


stage("guided-repair", repair, enabled=RUN_GPU_STAGES and RUN_REPAIR)

## 8. Evaluación (solo CPU, desde los registros guardados)

Todas las métricas se recalculan desde `results/`, así que esta celda se puede repetir sin GPU. Para cada
corrida, `compute_metrics.py` vuelve a extraer la letra de cada cadena con `extract_answer`, repite
`majority_vote`, compara con las respuestas correctas del manifiesto, ejecuta el evaluador oficial
`eval/evaluate.py` (global, por idioma y por materia), calcula el intervalo de Wilson al 95% y comprueba que los
ids sean los del subconjunto.

In [ ]:
RESULT_DIRS = [f"smoke-test/{n}" for n in CONFIGS] + list(CONFIGS) + ["guided-repair"]


def evaluate_all():
    done = []
    for name in RESULT_DIRS:
        if not (REPL / "results" / name / "raw").exists():
            print(f"{name}: sin registros crudos, se salta"); continue
        sh(f"source scripts/env.sh && python scripts/compute_metrics.py results/{name}"
           + (f" --expected {SMOKE_SIZE}" if name.startswith("smoke-test/") else ""))
        done.append(name)
    return f"{len(done)} carpetas de resultados"


stage("evaluation", evaluate_all)

## 9. Tablas comparativas, intervalos de confianza y ejemplos

`build_comparison.py` escribe `results/comparison.{csv,json,md}` y `results/by-language.{csv,json,md}`. La columna
del paper es la accuracy que reportan los autores sobre la **validación completa (4,651 preguntas)**: sirve de
contexto, no de meta. Con 200 preguntas el intervalo al 95% mide unos ±6 puntos, y por idioma (12–22 preguntas)
más de ±15, así que las diferencias pequeñas entre configuraciones o con el paper no se pueden ordenar con
este subconjunto.

In [ ]:
stage("comparison", lambda: sh("source scripts/env.sh && python scripts/build_comparison.py > /dev/null"))
stage("examples", lambda: sh("source scripts/env.sh && python scripts/export_examples.py"))

import pandas as pd
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
comparison = pd.read_csv(REPL / "results" / "comparison.csv")
# Tabla principal: accuracy con IC95, referencia del paper, fallas de extracción, tokens, tiempo, GPU y costo
comparison[["configuration", "accuracy_pct", "accuracy_ci95_low_pct", "accuracy_ci95_high_pct",
            "paper_full_val_accuracy_pct", "correct", "total", "question_extraction_failures",
            "chain_extraction_failure_rate", "truncated_chains", "generations_per_question",
            "avg_prompt_tokens_per_question_prefill", "avg_completion_tokens_per_question",
            "wall_clock_s", "avg_question_latency_s", "peak_gpu_memory_used_gib", "est_compute_usd",
            "verification_passed"]]

In [ ]:
# Accuracy por idioma: nuestras corridas frente a la Tabla 10 del paper (SC N=8, validación completa)
by_lang = pd.read_csv(REPL / "results" / "by-language.csv")
cols = ["language", "n_subset"] + [c for c in by_lang.columns if c.endswith("_pct") or c.endswith("_ci95")]
by_lang[cols]

In [ ]:
# Ejemplos para revisión: cuántos hay de cada tipo y uno de voto por mayoría con las 8 respuestas
examples = json.loads((REPL / "examples" / "examples.json").read_text())
print({k: len(v) for k, v in examples.items()})
ex = examples["majority_vote_all8"][0]
print(f"\nvoto por mayoría {ex['question_id']} ({ex['language']}, {ex['subject']}): correcta={ex['gold']} "
      f"predicha={ex['prediction']} votos={ex['vote_counts']}\nletras extraídas: {ex['extracted_answers']}")

## 10. Pruebas otra vez, validación de entrega y escaneo del paquete

In [ ]:
stage("tests-after", lambda: run_tests("after"), enabled=RUN_TESTS)


def validate():
    # Comprobaciones de entrega: mismo subconjunto, semilla 42, sin errores ocultos, métricas regenerables
    rows = []
    for name in comparison["configuration"]:
        m = metrics(name); v = m["verification"]
        meta = json.loads((REPL / "results" / name / "run_meta.json").read_text())
        rows.append({"result": name, "same_subset": v["ids_equal_expected_subset"],
                     "runner_subset_eq_manifest": v["runner_subset_indices_equal_manifest"],
                     "exit_code": meta.get("exit_code"), "logged_question_failures": len(v["runner_logged_question_failures"]),
                     "all_checks_passed": v["all_checks_passed"]})
    df = pd.DataFrame(rows); display(df)
    assert MANIFEST["seed"] == 42, "la semilla del subconjunto no es 42"
    assert df["same_subset"].all(), "alguna configuración usó otro subconjunto"
    assert df["runner_subset_eq_manifest"].isin([True, None]).all(), "el subconjunto del ejecutor no coincide con el manifiesto"
    assert (df["exit_code"] == 0).all() and (df["logged_question_failures"] == 0).all(), "alguna corrida reportó un error"
    assert df["all_checks_passed"].all(), "alguna métrica no se pudo regenerar desde los registros crudos"
    sh("bash scripts/scan_package.sh")   # credenciales, ids de cuenta, rutas personales, datasets, pesos
    return "todas las comprobaciones de entrega pasan"


stage("validation", validate)

## Fuera de alcance (no se ejecuta)

El notebook del equipo incluía dos fases más. No forman parte de esta tarea y aquí no se ejecutan:

- **Entrega al concurso ImageCLEF** (`configs/test/q35_sc_n16_2k_test.yaml`, N=16, 1,117 preguntas, ~4.5 h):
  el dataset `SU-FMI-AI/ImageCLEF-MR2026-MCQ-Visual` exige aceptar sus términos con una cuenta de Hugging Face
  (token), y las respuestas correctas son privadas, así que no se puede verificar.
- **Validación completa** (4,651 preguntas, ~19 h con N=16, más de $50): requiere aprobación explícita del costo.

Comandos oficiales, por si el equipo decide hacerlas:

```bash
python $REPO/scripts/experiment.py --config $REPO/configs/scaling/q35_sc_n8_2k.yaml          # validación completa
python $REPO/scripts/experiment.py --config $REPO/configs/test/q35_sc_n16_2k_test.yaml       # test ImageCLEF
python $REPO/scripts/repair_parse_failures.py --candidates runs/<run_id>/candidates.jsonl \
       --dataset imageclef_mcq_test --model Qwen/Qwen3.5-4B
python $REPO/scripts/format_submission.py --input runs/<run_id>/candidates.jsonl \
       --output submissions/test_submission.json --expected-count 1117
```

## 11. Copia a S3 y apagado

La copia usa `scripts/aws/s3sync.py` y el rol del espacio: no hay credenciales. El bucket debe ser privado
(Block Public Access) y cifrado. Con `STOP_APP_WHEN_DONE = True`, la app se borra solo si la copia salió bien,
y el kernel se detiene con ella.

In [ ]:
stage("s3-copy", sync_to_s3, enabled=bool(S3_BUCKET), note_disabled="S3_BUCKET vacío")
if WATCHDOG:
    WATCHDOG.cancel()

summary = pd.DataFrame(STATUS)
print(f"tiempo total del notebook: {(time.time() - NB_START) / 3600:.2f} h")
display(summary)
assert not (summary["status"] == "FAILED").any(), "alguna fase falló"

In [ ]:
if STOP_APP_WHEN_DONE and META.exists() and S3_BUCKET and any(r["stage"] == "s3-copy" and r["status"] == "ok" for r in STATUS):
    stop_this_app("notebook terminado")
else:
    print("la app sigue encendida. Apágala desde Studio, o usa STOP_APP_WHEN_DONE=True con S3_BUCKET, al terminar.")